# Speculative Decoding, Derived: Draft Models, N-grams, Medusa and EAGLE

The full code of [Speculative Decoding, Derived: Draft Models, N-grams, Medusa and EAGLE](https://residual-stream.lvtan71.workers.dev/posts/speculative-decoding/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/speculative-decoding) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/speculative-decoding.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 10 min end to end.
Downloads Qwen2.5-1.5B-Instruct and Qwen2.5-0.5B, about 4 GB.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*" "torch==2.14.*" "transformers==5.17.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "speculative-decoding"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `spec_math.py`

Checks by sampling that the acceptance rule reproduces the target's distribution with a bad draft; tokens per verification step, formula against simulation; speedup by draft cost.

In [ ]:
!{sys.executable} spec_math.py

The post printed:

```text
target p:           [0.4  0.25 0.15 0.1  0.07 0.03]
output frequencies: [0.4  0.25 0.15 0.1  0.07 0.03]
acceptance rate 0.550; theory: sum of min(p, q) = 0.550

alpha  k   formula  simulated
  0.6  2      1.96       1.96
  0.6  4      2.31       2.31
  0.6  8      2.47       2.48
  0.8  2      2.44       2.44
  0.8  4      3.36       3.36
  0.8  8      4.33       4.32
  0.9  2      2.71       2.71
  0.9  4      4.10       4.10
  0.9  8      6.13       6.13

speedup over plain decoding, by draft cost c (alpha = 0.8)
  k   c=0      c=0.05   c=0.1    c=0.3  
  1     1.80    1.71    1.64    1.38
  2     2.44    2.22    2.03    1.52
  4     3.36    2.80    2.40    1.53
  8     4.33    3.09    2.40    1.27
 16     4.89    2.72    1.88    0.84
```

## `accept.py`

Qwen2.5-1.5B-Instruct as the target, Qwen2.5-0.5B as the draft: per-position greedy agreement, sum of min(p, q), and two-token prompt lookup, on paragraph rewrites and open questions.

In [ ]:
!{sys.executable} accept.py

The post printed:

```text
rewrite a paragraph  (666 tokens): draft model, greedy 79%, sampled 75%; prompt lookup 42%
open question        (720 tokens): draft model, greedy 72%, sampled 70%; prompt lookup 6%
```

## `spec_steps.py`

The same models and prompts, measured as speculation runs: tokens per step for greedy speculation at k = 4 and 8 against the formula, acceptance on sampled text, and prompt lookup as deployed.

In [ ]:
!{sys.executable} spec_steps.py

The post printed:

```text
== rewrite: n=666; greedy agree 0.787 (per-prompt min 0.58 max 0.99)
   smin on greedy text 0.754; smin on T=1 sampled text 0.715
   lookup 1-token 2-gram 0.422
   greedy spec tokens/step k=4: actual 3.27, iid formula 3.28; k=8 actual 4.32, formula 4.15
   prompt lookup pl (tokens/step, frac steps with a proposal, frac proposals with >=1 accepted): (np.float64(2.937), np.float64(0.715), np.float64(0.456))
== open: n=720; greedy agree 0.718 (per-prompt min 0.68 max 0.76)
   smin on greedy text 0.702; smin on T=1 sampled text 0.722
   lookup 1-token 2-gram 0.060
   greedy spec tokens/step k=4: actual 2.82, iid formula 2.87; k=8 actual 3.34, formula 3.37
   prompt lookup pl (tokens/step, frac steps with a proposal, frac proposals with >=1 accepted): (np.float64(1.125), np.float64(0.403), np.float64(0.179))
```